In [ ]:
%py
# PySpark script for Databricks: Upsert metric_config and metric_master tables from Excel-derived CSVs
# Purpose: Ingest Excel data, validate, and upsert into purgo_playground.metric_config and purgo_playground.metric_master using Delta Lake MERGE
# Author: Giang Nguyen
# Date: 2025-09-29
# Description: Reads metric_config and metric_master data from CSVs (converted from Excel), validates schema and data quality, and performs upserts into Unity Catalog tables. Ensures new records are inserted and existing records are updated based on business keys.

# Import required PySpark modules
from pyspark.sql import functions as F  
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ShortType  
import logging  

# Set current catalog to Unity Catalog
spark.catalog.setCurrentCatalog("purgo_databricks")

# =========================
# Setup logging
# =========================
logger = logging.getLogger("metric_config_upsert")
logger.setLevel(logging.INFO)

# =========================
# Define schemas for metric_config and metric_master
# =========================
metric_config_schema = StructType([
    StructField("metric_id", StringType(), True),
    StructField("active_indicator", StringType(), True),
    StructField("geographical_average_type", StringType(), True),
    StructField("metric_data_type", StringType(), True),
    StructField("metric_description", StringType(), True),
    StructField("metric_template_name", StringType(), True),
    StructField("metric_type", StringType(), True),
    StructField("optional_filters", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("table_type", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("template_parameters", StringType(), True),
    StructField("bu_filter", StringType(), True),
    StructField("calling_service_name", StringType(), True),
    StructField("primary_key", StringType(), True),
])

metric_master_schema = StructType([
    StructField("metric_template_name", StringType(), True),
    StructField("sql_query", StringType(), True),
    StructField("metric_type", StringType(), True),
    StructField("view_name", StringType(), True),
    StructField("dependency", ShortType(), True),
])

# =========================
# Read CSV data from DBFS into DataFrames
# =========================
# NOTE: The Excel file must be converted to CSV files for each sheet: metric_config.csv and metric_master.csv
# Place these CSV files in dbfs:/FileStore/metric_config.csv and dbfs:/FileStore/metric_master.csv

metric_config_csv_path = "dbfs:/FileStore/metric_config.csv"
metric_master_csv_path = "dbfs:/FileStore/metric_master.csv"

def read_csv_to_df(file_path, schema):
    """
    Reads a CSV file from DBFS path into a DataFrame.

    Args:
        file_path (str): Path to the CSV file (absolute path)
        schema (StructType): Expected schema for the DataFrame

    Returns:
        DataFrame: Loaded DataFrame with the specified schema
    """
    try:
        df = (
            spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "false")
            .schema(schema)
            .load(file_path)
        )
        return df
    except Exception as e:
        logger.error(f"Failed to read CSV file {file_path}: {e}")
        raise

try:
    metric_config_df = read_csv_to_df(metric_config_csv_path, metric_config_schema)
except Exception as e:
    logger.error(f"Error loading metric_config CSV: {e}")
    raise

try:
    metric_master_df = read_csv_to_df(metric_master_csv_path, metric_master_schema)
except Exception as e:
    logger.error(f"Error loading metric_master CSV: {e}")
    raise

# =========================
# Data Quality Checks for metric_config
# =========================
def validate_metric_config(df):
    """
    Validates the metric_config DataFrame for required fields, data types, and duplicates.

    Args:
        df (DataFrame): metric_config DataFrame

    Returns:
        DataFrame: DataFrame of validation errors (empty if none)
    """
    errors = (
        df.withColumn("error_message",
            F.when(F.col("metric_id").isNull() | (F.trim(F.col("metric_id")) == ""), F.lit("Missing required field: metric_id"))
            .when(F.col("metric_template_name").isNull() | (F.trim(F.col("metric_template_name")) == ""), F.lit("Missing required field: metric_template_name"))
            .when(F.col("metric_type").isNull() | (F.trim(F.col("metric_type")) == ""), F.lit("Missing required field: metric_type"))
        )
    )
    dupes = (
        df.groupBy(F.col("metric_id"))
        .count()
        .filter(F.col("count") > 1)
        .withColumn("error_message", F.concat(F.lit("Duplicate metric_id "), F.col("metric_id"), F.lit(" in staging data")))
    )
    error_df = (
        errors.filter(F.col("error_message").isNotNull())
        .select("metric_id", "error_message")
        .unionByName(dupes.select("metric_id", "error_message"))
    )
    return error_df

# =========================
# Data Quality Checks for metric_master
# =========================
def validate_metric_master(df):
    """
    Validates the metric_master DataFrame for required fields, data types, and duplicates.

    Args:
        df (DataFrame): metric_master DataFrame

    Returns:
        DataFrame: DataFrame of validation errors (empty if none)
    """
    errors = (
        df.withColumn("error_message",
            F.when(F.col("metric_template_name").isNull() | (F.trim(F.col("metric_template_name")) == ""), F.lit("Missing required field: metric_template_name"))
            .when(F.col("sql_query").isNull() | (F.trim(F.col("sql_query")) == ""), F.lit("Missing required field: sql_query"))
            .when(F.col("metric_type").isNull() | (F.trim(F.col("metric_type")) == ""), F.lit("Missing required field: metric_type"))
            .when((~F.col("dependency").cast(ShortType()).isNotNull()) & F.col("dependency").isNotNull(), F.concat(F.lit("Invalid type for dependency: "), F.col("dependency")))
        )
    )
    dupes = (
        df.groupBy(F.col("metric_template_name"), F.col("dependency"))
        .count()
        .filter(F.col("count") > 1)
        .withColumn("error_message", F.concat(F.lit("Duplicate metric_master key (metric_template_name: "), F.col("metric_template_name"), F.lit(", dependency: "), F.col("dependency"), F.lit(") in staging data")))
    )
    error_df = (
        errors.filter(F.col("error_message").isNotNull())
        .select("metric_template_name", "dependency", "error_message")
        .unionByName(dupes.select("metric_template_name", "dependency", "error_message"))
    )
    return error_df

# =========================
# Validate and log errors for metric_config
# =========================
metric_config_errors = validate_metric_config(metric_config_df)
if metric_config_errors.count() > 0:
    logger.error("Validation errors in metric_config staging data:")
    metric_config_errors.show(truncate=False)
    raise Exception("Validation failed for metric_config staging data. See logs for details.")

# =========================
# Validate and log errors for metric_master
# =========================
metric_master_errors = validate_metric_master(metric_master_df)
if metric_master_errors.count() > 0:
    logger.error("Validation errors in metric_master staging data:")
    metric_master_errors.show(truncate=False)
    raise Exception("Validation failed for metric_master staging data. See logs for details.")

# =========================
# Enforce schema and column order for metric_config before upsert
# =========================
def enforce_metric_config_schema(df):
    """
    Enforces column order and types for metric_config DataFrame.

    Args:
        df (DataFrame): metric_config DataFrame

    Returns:
        DataFrame: DataFrame with columns in correct order and types
    """
    ordered_cols = [
        "metric_id",
        "active_indicator",
        "geographical_average_type",
        "metric_data_type",
        "metric_description",
        "metric_template_name",
        "metric_type",
        "optional_filters",
        "source_table",
        "table_type",
        "target_table",
        "template_parameters",
        "bu_filter",
        "calling_service_name",
        "primary_key"
    ]
    for f in metric_config_schema.fields:
        df = df.withColumn(f.name, F.col(f.name).cast(f.dataType))
    return df.select(ordered_cols)

# =========================
# Enforce schema and column order for metric_master before upsert
# =========================
def enforce_metric_master_schema(df):
    """
    Enforces column order and types for metric_master DataFrame.

    Args:
        df (DataFrame): metric_master DataFrame

    Returns:
        DataFrame: DataFrame with columns in correct order and types
    """
    ordered_cols = [
        "metric_template_name",
        "sql_query",
        "metric_type",
        "view_name",
        "dependency"
    ]
    for f in metric_master_schema.fields:
        df = df.withColumn(f.name, F.col(f.name).cast(f.dataType))
    return df.select(ordered_cols)

# =========================
# Prepare DataFrames for upsert
# =========================
metric_config_upsert_df = enforce_metric_config_schema(metric_config_df)
metric_master_upsert_df = enforce_metric_master_schema(metric_master_df)

# =========================
# Upsert into purgo_playground.metric_config using Delta Lake MERGE
# =========================
def upsert_metric_config(upsert_df):
    """
    Upserts records into purgo_playground.metric_config using Delta Lake MERGE.

    Args:
        upsert_df (DataFrame): DataFrame to upsert

    Returns:
        None
    """
    target_table = "purgo_databricks.purgo_playground.metric_config"
    staging_table = "purgo_databricks.purgo_playground.metric_config_stg"
    upsert_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(staging_table)
    merge_sql = f"""
    MERGE INTO {target_table} AS tgt
    USING {staging_table} AS src
    ON tgt.metric_id = src.metric_id
    WHEN MATCHED THEN
      UPDATE SET
        active_indicator = src.active_indicator,
        geographical_average_type = src.geographical_average_type,
        metric_data_type = src.metric_data_type,
        metric_description = src.metric_description,
        metric_template_name = src.metric_template_name,
        metric_type = src.metric_type,
        optional_filters = src.optional_filters,
        source_table = src.source_table,
        table_type = src.table_type,
        target_table = src.target_table,
        template_parameters = src.template_parameters,
        bu_filter = src.bu_filter,
        calling_service_name = src.calling_service_name,
        primary_key = src.primary_key
    WHEN NOT MATCHED THEN
      INSERT (
        metric_id,
        active_indicator,
        geographical_average_type,
        metric_data_type,
        metric_description,
        metric_template_name,
        metric_type,
        optional_filters,
        source_table,
        table_type,
        target_table,
        template_parameters,
        bu_filter,
        calling_service_name,
        primary_key
      )
      VALUES (
        src.metric_id,
        src.active_indicator,
        src.geographical_average_type,
        src.metric_data_type,
        src.metric_description,
        src.metric_template_name,
        src.metric_type,
        src.optional_filters,
        src.source_table,
        src.table_type,
        src.target_table,
        src.template_parameters,
        src.bu_filter,
        src.calling_service_name,
        src.primary_key
      )
    """
    spark.sql(merge_sql)

# =========================
# Upsert into purgo_playground.metric_master using Delta Lake MERGE
# =========================
def upsert_metric_master(upsert_df):
    """
    Upserts records into purgo_playground.metric_master using Delta Lake MERGE.

    Args:
        upsert_df (DataFrame): DataFrame to upsert

    Returns:
        None
    """
    target_table = "purgo_databricks.purgo_playground.metric_master"
    staging_table = "purgo_databricks.purgo_playground.metric_master_stg"
    upsert_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(staging_table)
    merge_sql = f"""
    MERGE INTO {target_table} AS tgt
    USING {staging_table} AS src
    ON tgt.metric_template_name <=> src.metric_template_name AND tgt.dependency <=> src.dependency
    WHEN MATCHED THEN
      UPDATE SET
        sql_query = src.sql_query,
        metric_type = src.metric_type,
        view_name = src.view_name
    WHEN NOT MATCHED THEN
      INSERT (
        metric_template_name,
        sql_query,
        metric_type,
        view_name,
        dependency
      )
      VALUES (
        src.metric_template_name,
        src.sql_query,
        src.metric_type,
        src.view_name,
        src.dependency
      )
    """
    spark.sql(merge_sql)

# =========================
# Execute upserts
# =========================
upsert_metric_config(metric_config_upsert_df)
upsert_metric_master(metric_master_upsert_df)

# =========================
# Success confirmation: Validate upserted data matches Excel data
# =========================
def confirm_metric_config_upsert():
    """
    Confirms that upserted metric_config data matches Excel data for key metric_ids.

    Returns:
        DataFrame: Result DataFrame for confirmation
    """
    key_ids = [
        "evenity_unit_hash", "customer_first_metric", "final_css", "batch_number"
    ]
    return (
        spark.table("purgo_databricks.purgo_playground.metric_config")
        .filter(F.col("metric_id").isin(key_ids))
        .select(
            "metric_id",
            "active_indicator",
            "metric_data_type",
            "metric_description",
            "metric_template_name",
            "metric_type",
            "source_table",
            "table_type",
            "target_table",
            "primary_key"
        )
    )

def confirm_metric_master_upsert():
    """
    Confirms that upserted metric_master data matches Excel data for key (metric_template_name, dependency).

    Returns:
        DataFrame: Result DataFrame for confirmation
    """
    key_pairs = [
        ("final_css", 2), ("evenity_unit_hash", 1), ("customer_first_metric", 1), ("batch_number", 1)
    ]
    cond = F.lit(False)
    for name, dep in key_pairs:
        cond = cond | ((F.col("metric_template_name") == name) & (F.col("dependency") == dep))
    return (
        spark.table("purgo_databricks.purgo_playground.metric_master")
        .filter(cond)
        .select(
            "metric_template_name",
            "sql_query",
            "metric_type",
            "view_name",
            "dependency"
        )
    )

# Show confirmation results
confirm_metric_config_upsert().show(truncate=False)
confirm_metric_master_upsert().show(truncate=False)

# End of script
